# Выбор генеративной модели

В блокноте сравниваются локальные языковые модели для генерации ответа по найденным фрагментам регламента. Главный критерий — понимание перефразированных вопросов: вопрос сформулирован другими словами, чем текст регламента, и модель должна понять, что фрагмент отвечает на него. Второй критерий — модель не должна выдумывать ответ, когда его нет во фрагментах (вопросы-ловушки).

Поиск во всех экспериментах одинаковый — лучший вариант из блокнота `parameter_tuning.ipynb`: гибрид RRF с переранжированием. Меняется только генеративная модель, поэтому разница в ответах объясняется моделью, а не поиском.

**Порядок работы:**

1. каждая модель отвечает на одни и те же 36 вопросов: 16 перефразированных, все 25 вопросов ТЗ (часть из них совпадает) и 3 ловушки; в контексте 5 фрагментов;
2. автоматически считаются доля распознанных ловушек, доля ложных отказов, наличие ссылок на фрагменты и время ответа;
3. ответы вручную размечаются по метрикам ТЗ: точность ответа 0/1/2 и наличие галлюцинаций;
4. для выбранной модели сравнивается качество ответов при 3, 5 и 10 фрагментах в контексте, как требует ТЗ.

**Подготовка.** Модели запускаются через [Ollama](https://ollama.com). Нужно установить Ollama, затем скачать модели командами `ollama pull <модель>` из списка ниже. Модели 7–8B в квантовании Q4 занимают около 5 ГБ и работают на процессоре при 16 ГБ оперативной памяти, но медленно: около 1–2 минут на вопрос. Ответы сохраняются в `data/generation/answers.csv` по мере получения, поэтому прерванный прогон продолжается с того же места.

In [ ]:
import re
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.evaluate import contains, load_queries
from src.generate import NOT_FOUND, answer, source_label
from src.hybrid_retrieval import build_hybrid

BLUE, ORANGE, AQUA, GREY = "#2a78d6", "#eb6834", "#1baf7a", "#9a9a9a"
plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb", "axes.edgecolor": "#c9c9c4",
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "grid.color": "#e6e6e1",
    "axes.axisbelow": True, "font.size": 10,
})

OUT_DIR = ROOT / "data" / "generation"
OUT_DIR.mkdir(parents=True, exist_ok=True)
ANSWERS_PATH = OUT_DIR / "answers.csv"

## 1. Модели и вопросы

Сравниваются модели одного класса — 3–8 млрд параметров, которые помещаются в 16 ГБ оперативной памяти:

- **Qwen2.5 7B** — многоязычная модель Alibaba, одна из сильнейших открытых моделей своего размера на русском языке;
- **Gemma 3 4B** — компактная многоязычная модель Google, проверяет, насколько хватает модели вдвое меньше;
- **YandexGPT 5 Lite 8B** — модель Яндекса, обученная в основном на русских текстах.

Список можно изменить: достаточно поправить `MODELS` и скачать модель через `ollama pull`.

In [ ]:
MODELS = {
    "Qwen2.5 7B": "qwen2.5:7b",
    "Gemma 3 4B": "gemma3:4b",
    "YandexGPT 5 Lite 8B": "hf.co/yandex/YandexGPT-5-Lite-8B-instruct-GGUF",
}

installed = {m["name"] for m in requests.get("http://localhost:11434/api/tags", timeout=10).json()["models"]}
for label, name in MODELS.items():
    ok = any(i == name or i.split(":")[0] == name.split(":")[0] for i in installed)
    print(f"{label:22s} {name:50s} {'скачана' if ok else 'НЕТ: ollama pull ' + name}")

In [ ]:
queries = [q for q in load_queries() if q["category"] in ("paraphrase", "trap") or q.get("tz")]
print(f"вопросов: {len(queries)}")
pd.Series([q["category"] for q in queries]).value_counts().rename("вопросов").to_frame()

In [ ]:
retriever = build_hybrid("rrf", rerank=True)

## 2. Прогон моделей

Для каждого вопроса сохраняются ответ, источники, время генерации и признак, есть ли ответ в переданных фрагментах (`context_has_answer`). Этот признак показывает, чья ошибка: если правильного фрагмента в контексте нет, модель не могла ответить, и это ошибка поиска, а не генерации.

In [ ]:
COLUMNS = ["model", "k", "query_id", "tz", "category", "question", "answer", "sources",
           "context_has_answer", "seconds", "accuracy", "hallucination"]


def load_answers() -> pd.DataFrame:
    if ANSWERS_PATH.exists():
        return pd.read_csv(ANSWERS_PATH, dtype={"tz": str}, keep_default_na=False)
    return pd.DataFrame(columns=COLUMNS)


def run(model_label: str, k: int = 5, num_ctx: int = 8192) -> None:
    done = load_answers()
    done_ids = set(done.loc[(done["model"] == model_label) & (done["k"].astype(int) == k), "query_id"])
    todo = [q for q in queries if q["query_id"] not in done_ids]
    for i, q in enumerate(todo, start=1):
        result = answer(q["query"], retriever, MODELS[model_label], k, num_ctx)
        texts = [hit.chunk["text"] for hit in result.sources]
        row = {
            "model": model_label, "k": k, "query_id": q["query_id"], "tz": q.get("tz", ""),
            "category": q["category"], "question": q["query"], "answer": result.text,
            "sources": " | ".join(source_label(hit) for hit in result.sources),
            "context_has_answer": int(any(contains(t, e) for t in texts for e in q["evidence"])),
            "seconds": round(result.seconds, 1), "accuracy": "", "hallucination": "",
        }
        pd.DataFrame([row], columns=COLUMNS).to_csv(
            ANSWERS_PATH, mode="a", header=not ANSWERS_PATH.exists(), index=False, encoding="utf-8")
        print(f"{model_label} k={k}: {i}/{len(todo)} {q['query_id']} {result.seconds:.0f} с")

In [ ]:
for label in MODELS:
    run(label, k=5)

## 3. Автоматические показатели

Без ручной разметки можно посчитать:

- **ловушки распознаны** — доля ловушек, на которые модель ответила «информация не найдена»;
- **ложные отказы** — доля вопросов с ответом в контексте, на которые модель всё равно ответила «информация не найдена»;
- **ответы со ссылками** — доля ответов, где есть ссылка на фрагмент вида [1];
- **время ответа** — среднее время генерации на процессоре.

In [ ]:
answers = load_answers()
answers["k"] = answers["k"].astype(int)
answers["context_has_answer"] = answers["context_has_answer"].astype(int)
answers["seconds"] = answers["seconds"].astype(float)
answers["refused"] = answers["answer"].str.lower().str.contains("не найден").astype(int)
answers["cited"] = answers["answer"].str.contains(r"\[\d+\]").astype(int)
main = answers[answers["k"] == 5]


def auto_metrics(df: pd.DataFrame) -> pd.Series:
    traps = df[df["category"] == "trap"]
    answerable = df[(df["category"] != "trap") & (df["context_has_answer"] == 1)]
    return pd.Series({
        "ловушки распознаны": traps["refused"].mean(),
        "ложные отказы": answerable["refused"].mean(),
        "ответы со ссылками": df.loc[df["category"] != "trap", "cited"].mean(),
        "время, с": df["seconds"].mean(),
    })


auto = main.groupby("model").apply(auto_metrics).reindex(list(MODELS))
auto.round(3)

In [ ]:
print("Доля вопросов, у которых ответ есть в 5 переданных фрагментах:")
answerable = main[main["category"] != "trap"].drop_duplicates("query_id")
answerable.groupby("category")["context_has_answer"].mean().round(2)

## 4. Ручная разметка ответов

Метрики ТЗ «точность ответа» и «доля галлюцинаций» считаются вручную. Для этого нужно открыть `data/generation/answers.csv` (в Excel, PyCharm или LibreOffice) и для каждой строки заполнить два столбца:

- `accuracy`: **2** — ответ верный и полный, **1** — частично верный, **0** — неверный или нет ответа; для ловушки 2 — модель ответила «информация не найдена», 0 — выдумала ответ;
- `hallucination`: **1** — в ответе есть факт, которого нет в переданных фрагментах, **0** — нет.

Правильный ответ можно сверить с фразами-доказательствами в `data/queries.jsonl` и со столбцом `sources`. Чтобы оценка была честной, лучше размечать, не глядя на столбец `model`, например скрыв его.

In [ ]:
answers = load_answers()
marked = answers[(answers["accuracy"] != "") & (answers["hallucination"] != "")].copy()
print(f"размечено: {len(marked)} из {len(answers)}")
marked["k"] = marked["k"].astype(int)
marked["accuracy"] = marked["accuracy"].astype(int)
marked["hallucination"] = marked["hallucination"].astype(int)
marked["paraphrase"] = marked["category"] == "paraphrase"


def manual_metrics(df: pd.DataFrame) -> pd.Series:
    para = df[df["paraphrase"]]
    return pd.Series({
        "точность, все (0–2)": df["accuracy"].mean(),
        "верных ответов, все": (df["accuracy"] == 2).mean(),
        "точность, перефразы (0–2)": para["accuracy"].mean(),
        "верных ответов, перефразы": (para["accuracy"] == 2).mean(),
        "галлюцинации": df["hallucination"].mean(),
    })


manual = marked[marked["k"] == 5].groupby("model").apply(manual_metrics).reindex(list(MODELS)).dropna(how="all")
manual.round(3) if len(manual) else "Ответы ещё не размечены: заполните accuracy и hallucination в answers.csv"

In [ ]:
def plot_manual(manual: pd.DataFrame) -> None:
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
    fig.suptitle("Качество ответов моделей (5 фрагментов в контексте)", x=0.01, ha="left", fontsize=12)
    y = np.arange(len(manual))
    axes[0].barh(y - 0.2, manual["верных ответов, все"], 0.4, color=BLUE, label="все вопросы")
    axes[0].barh(y + 0.2, manual["верных ответов, перефразы"], 0.4, color=ORANGE, label="перефразы")
    axes[0].set_title("Доля верных ответов", loc="left", fontsize=10, color="#52514e")
    axes[1].barh(y, manual["галлюцинации"], 0.5, color=GREY)
    axes[1].set_title("Доля ответов с галлюцинациями", loc="left", fontsize=10, color="#52514e")
    for ax in axes:
        ax.set_yticks(y, manual.index)
        ax.set_xlim(0, 1)
    axes[0].invert_yaxis()
    axes[1].set_yticklabels([])
    fig.legend(*axes[0].get_legend_handles_labels(), loc="upper right", ncol=2, frameon=False)
    plt.tight_layout()


if len(manual):
    plot_manual(manual)
else:
    print("Сначала разметьте ответы")

**Вывод.** _Заполняется после разметки: какая модель лучше отвечает на перефразированные вопросы, какая реже выдумывает факты и распознаёт ловушки, насколько это стоит по времени._

## 5. Число фрагментов в контексте

ТЗ требует сравнить качество ответов при 3, 5 и 10 фрагментах. Сравнение делается для выбранной модели. Больше фрагментов — выше шанс, что ответ есть в контексте, но модели сложнее найти его среди лишнего текста, и ответ генерируется дольше. Для 10 фрагментов контекстное окно увеличено до 16 тыс. токенов.

In [ ]:
BEST_MODEL = "Qwen2.5 7B"  # выбрать по результатам раздела 4

run(BEST_MODEL, k=3)
run(BEST_MODEL, k=10, num_ctx=16384)

In [ ]:
answers = load_answers()
answers["k"] = answers["k"].astype(int)
answers["context_has_answer"] = answers["context_has_answer"].astype(int)
answers["seconds"] = answers["seconds"].astype(float)
best = answers[answers["model"] == BEST_MODEL]
by_k = best.groupby("k").agg(
    **{"ответ есть в контексте": ("context_has_answer", "mean"), "время, с": ("seconds", "mean")})
marked_best = best[(best["accuracy"] != "") & (best["hallucination"] != "")].copy()
if len(marked_best):
    marked_best["accuracy"] = marked_best["accuracy"].astype(int)
    marked_best["hallucination"] = marked_best["hallucination"].astype(int)
    by_k = by_k.join(marked_best.groupby("k").agg(
        **{"верных ответов": ("accuracy", lambda s: (s == 2).mean()),
           "галлюцинации": ("hallucination", "mean")}))
by_k.round(3)

**Вывод.** _Заполняется после прогона и разметки: при каком числе фрагментов ответы лучше и стоит ли увеличение контекста потраченного времени._

## Итог

_Заполняется в конце: выбранная модель, число фрагментов и обоснование._